In [10]:
# importing pandas so i can read and analyse the csv file
import pandas as pd

# loading the csv file into pandas - forms dataframe
df = pd.read_csv('/content/telemetry_data(in).csv')

# printing basic info to check the data loaded correctly
print("Data loaded successfully!")
print(f"Total readings found: {len(df)}")
print(f"Turbines monitored: {df['turbine_id'].nunique()}")
print("-" * 45)

# grouping all readings by turbine and calculating average temp and highest vibration
stats = df.groupby('turbine_id').agg(
    avg_temperature=('temperature_c', 'mean'),
    max_vibration=('vibration_mm_s', 'max')
).round(2)

# printing the summary so i can see all turbines at a glance and help identify anomalies easier
print("\nSummary stats per turbine:")
print(stats)
print("-" * 45)

# setting the safety parameters based on the anomaly rules in the brief
TEMP_LIMIT = 85.0
VIBRATION_LIMIT = 15.0

# filtering for any turbine that breaks either rule
failing_turbines = stats[
    (stats['avg_temperature'] > TEMP_LIMIT) |
    (stats['max_vibration'] > VIBRATION_LIMIT)
]

# printing the results
print("  TURBINES REQUIRING URGENT MAINTENANCE")

for turbine_id, row in failing_turbines.iterrows():
    print(f"\nTurbine ID: {turbine_id}")
    if row['avg_temperature'] > TEMP_LIMIT:
        print(f"  [WARNING] HIGH TEMPERATURE: Average = {row['avg_temperature']}°C (Limit: {TEMP_LIMIT}°C)")
    if row['max_vibration'] > VIBRATION_LIMIT:
        print(f"  [WARNING] HIGH VIBRATION: Max spike = {row['max_vibration']} mm/s (Limit: {VIBRATION_LIMIT} mm/s)")

print(f"Total turbines flagged: {len(failing_turbines)}")

Data loaded successfully!
Total readings found: 5000
Turbines monitored: 10
---------------------------------------------

Summary stats per turbine:
            avg_temperature  max_vibration
turbine_id                                
T-01                  65.32           11.9
T-02                  65.16           12.0
T-03                  65.31           12.0
T-04                  90.58           10.0
T-05                  64.91           12.0
T-06                  64.80           12.0
T-07                  67.78           25.0
T-08                  64.86           12.0
T-09                  65.44           12.0
T-10                  64.41           12.0
---------------------------------------------
  TURBINES REQUIRING URGENT MAINTENANCE

Turbine ID: T-04
  [WARNING] HIGH TEMPERATURE: Average = 90.58°C (Limit: 85.0°C)

Turbine ID: T-07
  [WARNING] HIGH VIBRATION: Max spike = 25.0 mm/s (Limit: 15.0 mm/s)
Total turbines flagged: 2
